# Canonical attacks evaluated on AnomalyCLIP

This notebook is an independent Kaggle entry point for AnomalyCLIP. It clones the experiment and official model repositories, reads the selected per-dataset, per-category, or per-image attacks from the attached canonical Kaggle dataset, uses the CSV's fixed evaluation IDs, and reports clean/adversarial metrics. Enable a GPU and Internet before running all cells.

In [ ]:
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE REPOSITORIES AND INSTALL DEPENDENCIES =====')
WORKING = Path('/kaggle/working')
EXPERIMENT_ROOT = WORKING / 'adversarial-robustness'
ANOMALYCLIP_ROOT = WORKING / 'AnomalyCLIP'
EXPERIMENT_REPO_URL = 'https://github.com/Parsagh05/adversarial-robustness.git'
ANOMALYCLIP_REPO_URL = 'https://github.com/zqhang/AnomalyCLIP.git'
# This is the official AnomalyCLIP revision recorded by the canonical artifacts.
ANOMALYCLIP_COMMIT = '3911738c0867544f545a076ad78f3f11d9ecbfdf'

def clone_or_update(url, destination, commit=None):
    if destination.exists():
        subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    else:
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    if commit:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    else:
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)

clone_or_update(EXPERIMENT_REPO_URL, EXPERIMENT_ROOT)
clone_or_update(ANOMALYCLIP_REPO_URL, ANOMALYCLIP_ROOT, ANOMALYCLIP_COMMIT)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'requirements.txt')
], check=True)
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))
print('Experiment code:', EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline')
print('Official AnomalyCLIP:', ANOMALYCLIP_ROOT)

In [ ]:
import shutil

from blackbox_evaluation_pipeline.universal_eval.artifacts import load_manifest

print('===== STEP 2: RESOLVE THE ATTACHED CANONICAL ATTACK DATASET =====')
# Selection options: use one or more scopes and datasets. Set a filter to None for all.
ATTACK_SCOPES = ('per_dataset',)  # per_dataset, per_category, per_image
ATTACK_DATASETS = ('mvtec', 'visa')  # use ('visa',) or ('mvtec',) for one dataset
ATTACK_CATEGORIES = None
ATTACK_DIRECTIONS = None  # e.g. ('normal_to_abnormal',)
ATTACK_LOSS_MODES = None  # e.g. ('global', 'local', 'combined')

def valid_artifact_root(path):
    return path.is_dir() and all(
        (path / f'canonical_clip_{scope}' / 'attack_manifest.csv').is_file()
        for scope in ATTACK_SCOPES
    )

candidates = [
    Path('/kaggle/input/datasets/alirezasalehy/adversarial-attacks-vlm-survey'),
    Path('/kaggle/input/adversarial-attacks-vlm-survey'),
]
if Path('/kaggle/input').is_dir():
    candidates.extend(path.parent for path in Path('/kaggle/input').rglob('canonical_clip_per_dataset'))
ARTIFACTS_ROOT = next((path for path in candidates if valid_artifact_root(path)), None)
if ARTIFACTS_ROOT is None:
    raise FileNotFoundError(
        'Attach alirezasalehy/adversarial-attacks-vlm-survey as a Kaggle input.'
    )

artifacts = load_manifest(
    ARTIFACTS_ROOT, scopes=ATTACK_SCOPES, sources=ATTACK_DATASETS,
    targets=ATTACK_DATASETS, categories=ATTACK_CATEGORIES,
    directions=ATTACK_DIRECTIONS, loss_modes=ATTACK_LOSS_MODES,
)
print('Canonical root:', ARTIFACTS_ROOT)
print('Available conditions:', len(artifacts))
for source, target in sorted({(a.record['source_dataset'], a.record['target_dataset']) for a in artifacts}):
    count = sum(a.record['source_dataset'] == source and a.record['target_dataset'] == target for a in artifacts)
    print(f'  {source} -> {target}: {count}')

In [ ]:
import torch

print('===== STEP 3: RESOLVE DATASETS AND TARGET CHECKPOINTS =====')

def first_existing_directory(paths, label):
    for path in paths:
        if path.is_dir():
            return path
    raise FileNotFoundError(f'{label} was not found. Checked: {paths}')

MVTEC_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection'),
    Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'),
], 'MVTec AD')
VISA_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922'),
    Path('/kaggle/input/visa-ad/VisA_20220922'),
], 'VisA')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')

# Opposite-dataset checkpoints preserve the original zero-shot protocol.
MVTEC_TARGET_CHECKPOINT = ANOMALYCLIP_ROOT / 'checkpoints' / '9_12_4_multiscale' / 'epoch_15.pth'
VISA_TARGET_CHECKPOINT = ANOMALYCLIP_ROOT / 'checkpoints' / '9_12_4_multiscale_visa' / 'epoch_15.pth'
for checkpoint in (MVTEC_TARGET_CHECKPOINT, VISA_TARGET_CHECKPOINT):
    if not checkpoint.is_file():
        available = sorted((ANOMALYCLIP_ROOT / 'checkpoints').rglob('*.pth'))
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint}. Available: {available}')

MODEL_KWARGS_BY_TARGET = {
    'mvtec': {
        'repository_root': str(ANOMALYCLIP_ROOT),
        'checkpoint_path': str(MVTEC_TARGET_CHECKPOINT),
        'clip_download_root': str(WORKING / 'clip_cache'),
    },
    'visa': {
        'repository_root': str(ANOMALYCLIP_ROOT),
        'checkpoint_path': str(VISA_TARGET_CHECKPOINT),
        'clip_download_root': str(WORKING / 'clip_cache'),
    },
}
print('MVTec:', MVTEC_ROOT)
print('VisA:', VISA_ROOT)
print('MVTec target checkpoint:', MVTEC_TARGET_CHECKPOINT)
print('VisA target checkpoint:', VISA_TARGET_CHECKPOINT)

In [ ]:
from blackbox_evaluation_pipeline import EvaluationConfig, run_evaluation

print('===== STEP 4: RUN FIXED-ID CLEAN/ADVERSARIAL EVALUATION =====')
FULL_RUN = True
OUTPUT_ROOT = WORKING / (
    'kaggle_new_anomalyclip_full' if FULL_RUN else 'kaggle_new_anomalyclip_check'
)
SAMPLES_ROOT = WORKING / (
    'kaggle_new_anomalyclip_samples_full' if FULL_RUN else 'kaggle_new_anomalyclip_samples_check'
)
THRESHOLDS_BY_TARGET = {
    dataset: str(EXPERIMENT_ROOT / 'attack_generation_pipeline' / 'thresholds' / dataset / 'category_thresholds.json')
    for dataset in ('mvtec', 'visa')
}
for dataset, threshold_path in THRESHOLDS_BY_TARGET.items():
    if not Path(threshold_path).is_file():
        raise FileNotFoundError(f'Missing frozen {dataset} thresholds: {threshold_path}')

config = EvaluationConfig(
    artifacts_root=str(ARTIFACTS_ROOT),
    mvtec_root=str(MVTEC_ROOT),
    visa_root=str(VISA_ROOT),
    output_root=str(OUTPUT_ROOT),
    model_name='anomalyclip',
    model_kwargs_by_target=MODEL_KWARGS_BY_TARGET,
    thresholds_by_target=THRESHOLDS_BY_TARGET,
    device='cuda',
    batch_size=2,
    metric_size=518,
    anomaly_map_sigma=4.0,
    aupro_fpr_limit=0.30,
    aupro_max_thresholds=200,
    verify_checksums=True,
    save_predictions=True,
    save_qualitative_samples=True,
    qualitative_output_root=str(SAMPLES_ROOT),
    attack_scopes=ATTACK_SCOPES,
    source_datasets=ATTACK_DATASETS,
    target_datasets=ATTACK_DATASETS,
    attack_categories=ATTACK_CATEGORIES,
    attack_directions=ATTACK_DIRECTIONS,
    attack_loss_modes=ATTACK_LOSS_MODES,
    max_conditions=None if FULL_RUN else 1,
    run_notes='Attached canonical CSV attack bundles; fixed evaluation IDs.',
)
SUMMARY_PATH = run_evaluation(config)
print('Finished:', SUMMARY_PATH)

In [ ]:
import csv

print('===== STEP 5: PREVIEW SUMMARY =====')
with SUMMARY_PATH.open(newline='', encoding='utf-8') as handle:
    summary_rows = list(csv.DictReader(handle))
columns = [
    'source_dataset', 'target_dataset', 'scope', 'category', 'direction', 'loss_mode',
    'clean_i_auroc', 'adversarial_i_auroc', 'delta_i_auroc',
    'clean_p_auroc', 'adversarial_p_auroc', 'delta_p_auroc',
    'clean_aupro', 'adversarial_aupro', 'delta_aupro',
    'clean_accuracy', 'adversarial_accuracy',
    'clean_fpr', 'adversarial_fpr', 'clean_fnr', 'adversarial_fnr',
    'attack_flip_rate', 'targeted_attack_success_rate',
]
for row in summary_rows:
    print({column: row[column] for column in columns})

In [ ]:
print('===== STEP 6: PACKAGE OUTPUTS =====')
results_archive = shutil.make_archive(
    str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name
)
samples_archive = shutil.make_archive(
    str(SAMPLES_ROOT), 'zip', root_dir=SAMPLES_ROOT.parent, base_dir=SAMPLES_ROOT.name
)
print('Packaged numerical results:', results_archive)
print('Packaged qualitative samples:', samples_archive)